# Rider Prediction Tutorial

This notebook demonstrates how to use Rider for predictions. We will configure the necessary paths, run the core `rider-predict` command, and intuitively visualize the output results.

In [1]:
import os
import glob
import pandas as pd

# Assume this Jupyter Notebook is running in the Rider project root directory.
# If not, please change BASE_DIR to your absolute path, e.g., '/root/gaoyang/westlake_emblab/Rider'
BASE_DIR = os.getcwd() 

# Define input, output, and model weight paths
INPUT_DIR = os.path.join(BASE_DIR, "test_data")
OUTPUT_PATH = os.path.join(INPUT_DIR, "test_results")
WEIGHTS = os.path.join(BASE_DIR, "checkpoint/checkpoint-19600/model.safetensors")
RDRP_DB = os.path.join(BASE_DIR, "Rider_RDSDB_30/pdbs")
SUBMODULE_DIR = os.path.join(BASE_DIR, "submodule")

# Ensure the output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Using weights from: {WEIGHTS}")
print(f"Input dir: {INPUT_DIR}")
print(f"Output dir: {OUTPUT_PATH}")
print(f"RDRP DB: {RDRP_DB}")

Using weights from: /root/gaoyang/westlake_emblab/Rider/checkpoint/checkpoint-19600/model.safetensors
Input dir: /root/gaoyang/westlake_emblab/Rider/test_data
Output dir: /root/gaoyang/westlake_emblab/Rider/test_data/test_results
RDRP DB: /root/gaoyang/westlake_emblab/Rider/Rider_RDSDB_30/pdbs


In [2]:
# Let's take a look at the input file
test_file = os.path.join(INPUT_DIR, "test_AJ004930_1.faa")

print(f"--- Previewing input file: {os.path.basename(test_file)} ---")
with open(test_file, 'r') as f:
    # Print the first 10 lines
    lines = f.readlines()
    for line in lines[:10]:
        print(line.strip())
    if len(lines) > 10:
        print("...")

--- Previewing input file: test_AJ004930_1.faa ---
>lcl|AJ004930.1_prot_CAA06228.1_1 [db_xref=GOA:O92614,InterPro:IPR008686,UniProtKB/TrEMBL:O92614] [protein=RNA-dependent RNA polymerase] [protein_id=CAA06228.1] [location=269..2425] [gbkey=CDS]
MKRLTLSQNKSNQLTNNDLSNVGYITKQLFPHWIRLLVWSLQLSPAPYKKFGSRIAILWKANGVSFTVQY
LKECTRIVQHFVSGHPVFVTDVMPIGLAGGLPTIIPGTLRTLLRSKDSSTIRGVLSTLAVYRIMKMPCVL
KLESITDPFKGISDTLPKSEIINGLASLGFEIPKGRSKHLLTLSNPIIYLLSAGPNHSISMMGIWKDIYA
WYVSPLFPTLLSFIGRMNRGNVLIDLLRAEVSYWEATGVKPSVSPLDLKLGKLAIKEEAAGKARVFAMAD
SITQSVMAPLNSWVFSKLKDLPMDGTFNQQAPLNRLVQLYQDGLLHDVEFYSYDLSSATDRLPMAFQKQI
ISVLFGSKFAKDWATLLVGRDWYLKDIPYRYSVGQPMGALSSWAMLALSHHVIVQIAAMRVGKLPFTNYA
LLGDDIVIADKAVATSYHMIMTQILGVEINLSKSLVSNNSFEFAKRLVTMDGEVSAVGAKNLLVALKSRW
GISSVILDLYNKGLALSEQDLRQRFSSIPTVSKQFGVDKLLWLVLGPFGFIPSKDGLSAFMKLNRSLSLV
DMHILLSCVDEAKFDLDKKTWEANIQETVHTLLRFGMLSEPAGFEVFSDFTSSPLYSFIRGQFGNKLSAL
...


## Running the Rider Prediction

Next, we will execute the `rider-predict` command. In Jupyter, we can use `!` to run shell commands and `{}` to pass Python variables directly into the command.

In [5]:
# Specify which GPU to use
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# Run the prediction command
!rider-predict \
    -i {test_file} \
    -t 32 \
    -w {WEIGHTS} \
    -b 1 \
    --device cuda \
    -o {OUTPUT_PATH} \
    --threads 32 \
    --submodule_dir {SUBMODULE_DIR} \
    --predict_structure \
    --sequence_length 1024 \
    --structure_align_enabled \
    --rdrp_structure_database {RDRP_DB} \
    --prob_threshold 50 \
    --threshold_type 4 \
    --top_n_mean_prob 5 \
    --alignment-type 1 \
    --threshold 0.9

print("Prediction finished!")

Updated PATH: /root/gaoyang/westlake_emblab/Rider/submodule/foldseek/bin:/opt/miniforge3/envs/rider/bin:/opt/miniforge3/share/rubygems/bin:/usr/storage/yuhao/cryosparc/cryosparc_master/bin:/opt/miniforge3/share/rubygems/bin:/usr/local/texlive/2019/bin/x86_64-linux:/root/.vscode-server/cli/servers/Stable-e7fb5e96c0730b9deb70b33781f98e2f35975036/server/bin/remote-cli:/usr/commondata/public/zhaoyeping/projects/saw/saw-8.1.3/bin:/usr/commondata/public/zhaoyeping/projects/cellranger/cellranger-10.0.0:/root/.nvm/versions/node/v22.22.1/bin:/root/.local/bin:/opt/miniforge3/envs/rider/bin:/root/miniforge3/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin
2026-04-03 18:21:18,663 - INFO - Logging to file: /root/gaoyang/westlake_emblab/Rider/test_data/test_results/test_AJ004930_1.faa/test_AJ004930_1.faa_intermediate/rider_pipeline.log
2026-04-03 18:21:18,663 - INFO - Processing file: /root/gaoyang/westlake_emblab/Rider/test_data/test_AJ0049

## Understanding the Output Files

Once the prediction is complete, Rider generates several output files in the specified output directory. The results are organized in a subfolder named after the input file. 

Among the generated files, there are two key text files that summarize the taxonomy and alignment results:
1. `final_result_taxonomy_id.txt`: Contains the basic filtering results based on your specified thresholds.
2. `final_result_taxonomy_id_with_seq.txt`: A high-quality subset of the first file, which includes strict length/score filtering and appends the actual amino acid sequences.

Let's load and inspect these files using `pandas`.

In [6]:
from IPython.display import display

# The output folder is named after the input file
input_basename = os.path.basename(test_file)
result_dir = os.path.join(OUTPUT_PATH, input_basename)

file1_path = os.path.join(result_dir, "final_result_taxonomy_id.txt")
file2_path = os.path.join(result_dir, "final_result_taxonomy_id_with_seq.txt")

print(f"Result directory: {result_dir}")
print(f"Checking if output files exist...")
print(f"File 1 exists: {os.path.exists(file1_path)}")
print(f"File 2 exists: {os.path.exists(file2_path)}")

Result directory: /root/gaoyang/westlake_emblab/Rider/test_data/test_results/test_AJ004930_1.faa
Checking if output files exist...
File 1 exists: True
File 2 exists: True


### 1. Basic Taxonomy Results (`final_result_taxonomy_id.txt`)

This file contains all alignments that met the initial probability threshold (e.g., `bits >= 50` or `TM-score >= 0.5`, depending on your `--threshold_type` setting). 

**Columns:**
- **Query_ID**: The ID of the query sequence (with the `Rider_` prefix removed).
- **Target_ID**: The matched target sequence ID in the database.
- **Bits_Score**: Alignment bit score (higher is better).
- **qTM_score**: TM-score normalized by the query length.
- **tTM_score**: TM-score normalized by the target length.
- **LDDT**: Local Distance Difference Test score, evaluating local structural accuracy.
- **E_value**: Expectation value.

In [7]:
# Define column names for the first file
cols_file1 = [
    "Query_ID", "Target_ID", "Bits_Score", 
    "qTM_score", "tTM_score", "LDDT", "E_value"
]

if os.path.exists(file1_path):
    print("=== Basic Taxonomy Results ===")
    df1 = pd.read_csv(file1_path, sep='\t', names=cols_file1)
    display(df1.head())
else:
    print("Basic taxonomy result file not found.")

=== Basic Taxonomy Results ===


,Query_ID,Target_ID,Bits_Score,qTM_score,tTM_score,LDDT,E_value
0,AJ004930.1_2_1;1-90,Rv4_040347,74,0.7991,0.1936,0.78,0.483
1,AJ004930.1_3_1;1-95,Rv4_142573,86,0.8984,0.2294,0.85,0.529
2,lcl|AJ004930.1_prot_CAA06228.1_1_1;1-718,Rv4_159529,48,0.4855,0.8688,0.61,0.658


### 2. High-Quality Results with Sequences (`final_result_taxonomy_id_with_seq.txt`)

This file is a strictly filtered subset of the previous file. It applies a dual-filtering logic based on sequence length and alignment score to ensure high reliability:
- Length $\ge$ 100 AND Score $\ge$ Threshold 1 (e.g., Bits $\ge$ 50)
- Length between 80-99 AND Score $\ge$ Threshold 2 (e.g., Bits $\ge$ 60)

*Note: Alignments that fail these strict criteria are saved in a separate `_low_quality.txt` file.*

**Additional Columns:**
- **Cleaned_Query_ID**: The base ID of the query (coordinates removed).
- **qstart / qend**: The start and end positions of the query in the alignment.
- **start_pos / end_pos**: The parsed coordinates from the original ID string.
- **Sequence**: The full amino acid sequence extracted from the raw input.

In [8]:
# Define column names for the second file
cols_file2 = [
    "Cleaned_Query_ID", "Target_ID", "Bits_Score", 
    "qTM_score", "tTM_score", "LDDT", "E_value", 
    "qstart", "qend", "start_pos", "end_pos", "Sequence"
]

if os.path.exists(file2_path):
    print("=== High-Quality Results with Sequences ===")
    df2 = pd.read_csv(file2_path, sep='\t', names=cols_file2)
    
    # Truncate the sequence column for better display in Jupyter
    df2_display = df2.copy()
    df2_display['Sequence'] = df2_display['Sequence'].apply(lambda x: x[:30] + "..." if isinstance(x, str) and len(x) > 30 else x)
    
    display(df2_display.head())
else:
    print("High-quality sequence result file not found.")

=== High-Quality Results with Sequences ===


,Cleaned_Query_ID,Target_ID,Bits_Score,qTM_score,tTM_score,LDDT,E_value,qstart,qend,start_pos,end_pos,Sequence
0,AJ004930.1_2,Rv4_040347,74,0.7991,0.1936,0.78,0.483,1,90,1,90,MDGTFNQQAPLNRLVQLYQDGLLHDVEFYS...
1,AJ004930.1_3,Rv4_142573,86,0.8984,0.2294,0.85,0.529,1,95,1,95,MLALSHHVIVQIAAMRVGKLPFTNYALLGD...
2,lcl|AJ004930.1_prot_CAA06228.1_1,Rv4_159529,48,0.4855,0.8688,0.61,0.658,29,514,1,718,MKRLTLSQNKSNQLTNNDLSNVGYITKQLF...
